In [17]:
import datetime
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler,MaxAbsScaler
from sklearn.decomposition import KernelPCA
import numpy as np
import pandas as pd
import math
import keras
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.callbacks import EarlyStopping
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from keras.layers import Attention

tf.random.set_seed(666)


In [18]:
# Single step dataset preparation
def singleStepSampler(df, window):
	xRes = []
	yRes = []
	for i in range(0, len(df) - window):
		res = []
		for j in range(0, window):
			r = []
			for col in df.columns:
				r.append(df[col][i + j])
			res.append(r)
		xRes.append(res)
		yRes.append(df[['dis']].iloc[i + window].values)
	return np.array(xRes), np.array(yRes)


In [19]:
# Dataset loading
name = '2_dsk_imputMF_half'
csv_path = f"F:\geodata\\river_runoff_obs\\{name}.csv"
csv_path = r"C:\Users\HP\Desktop\2_dsk_imputMF_half.csv"
usecols = ['time', 'pre', 'tm', 'tmax', 'tmin',  'dis']
dataFrame_full = pd.read_csv(csv_path, usecols=usecols)
# this refers to observed runoff and 10d_mean refers to the runoff with MA
dataFrame_full.head()

,time,pre,tm,tmax,tmin,dis
0,1968/6/1,0.826992,17.866158,24.362043,11.738063,120.5
1,1968/6/2,0.608511,17.439476,24.046434,10.989391,127.1
2,1968/6/3,7.152899,14.242048,21.398450,11.116452,135.9
3,1968/6/4,0.965721,12.069605,17.970097,6.217866,150.1
4,1968/6/5,0.000000,14.461229,21.663729,5.666883,180.8


In [20]:
# Get Index of NaN Values in a Specific Column:

# Create a boolean series that identifies NaN values
is_null = dataFrame_full['dis'].isnull()


# Shift the boolean series to identify changes between NaN and non-NaN values
shifted = is_null.ne(is_null.shift()).cumsum()

# Group by the shifted series and filter to only NaN sections
nan_sections = dataFrame_full[is_null].groupby(shifted)

# Extract the start and end indices for each section and create separate lists for them
start_indices = []
end_indices = []

for _, group in nan_sections:
    start_indices.append(group.index[0])
    end_indices.append(group.index[-1])

# Create a DataFrame from the lists
null_index_df = pd.DataFrame({'Start': start_indices, 'End': end_indices})

In [21]:
null_index_df

,Start,End
0,8249,11170


In [22]:
def intercept_data(i):
	global dataFrame
	# Intercept the first section of missing value
	# dataFrame = dataFrame_full[ :null_index_df["End"][i]+365]
	dataFrame = dataFrame_full[ :null_index_df["End"][i]+1]

In [23]:
def scaler_data(dataFrame ):
	global scaler
	imputer = SimpleImputer(missing_values=np.nan) # Handling missing values
	if 'time' in dataFrame.columns:
		dataFrame.drop(columns=['time'], inplace=True)
	dataFrame = pd.DataFrame(imputer.fit_transform(dataFrame), columns=dataFrame.columns)
	dataFrame = dataFrame.reset_index(drop=True)
	# Applying feature scaling
	scaler = MinMaxScaler(feature_range=(0, 1))
	df_scaled = scaler.fit_transform(dataFrame.to_numpy())
	df_scaled = pd.DataFrame(df_scaled, columns=list(dataFrame.columns))
	target_scaler = MinMaxScaler(feature_range=(0, 1))
	df_scaled[ ['dis']] = target_scaler.fit_transform(dataFrame[ ['dis']].to_numpy())
	df_scaled = df_scaled.astype(float)
	
	return df_scaled

In [24]:
## Data spliting

In [25]:
time_step = 365

In [26]:
def split_data(df_scaled ,i):

	# Dataset splitting
	SPLIT = 0.7 # Equal to the rate by Pr. Wang Lei train_data_rate = 0.7

	(xVal, yVal) = singleStepSampler(df_scaled, time_step)
	X_train =    xVal[:int(SPLIT * len(xVal))]
	y_train =    yVal[:int(SPLIT * len(yVal))]
	X_test =     xVal[int(SPLIT * len(xVal)):]
	y_test =     yVal[int(SPLIT * len(yVal)):]
	
	X_forecast = xVal[null_index_df["Start"][i]-time_step  :null_index_df["End"][i]+1] # For singleStepSampler, the time step need to be added.
	# X_forecast = xVal[:] # We will use the whole dataset for forecast
	yVal[null_index_df["Start"][i]-time_step  :null_index_df["End"][i]+1] = 0
	"""Whether or not as the following"""
	# X_forecast = xVal[null_index_df["Start"][i]-time_step  :null_index_df["End"][i]+1] # For singleStepSampler, the time step need to be added.

	return X_train, y_train, X_test, y_test ,X_forecast

In [27]:
# model source https://www.nature.com/articles/s41598-024-63989-7#Sec2
import tensorflow as tf
from tensorflow.keras import layers, models

def model_building(X_train):
    multivariate_lstm = models.Sequential()
    
    # First LSTM layer with 512 neurons and ReLU activation
    multivariate_lstm.add(layers.LSTM(units=512, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Second LSTM layer with a progressive reduction in neurons
    multivariate_lstm.add(layers.LSTM(units=256, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Third LSTM layer with further reduction in neurons
    multivariate_lstm.add(layers.LSTM(units=128, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Fourth LSTM layer with 64 neurons
    multivariate_lstm.add(layers.LSTM(units=64, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Fifth LSTM layer with 32 neurons
    multivariate_lstm.add(layers.LSTM(units=32, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Sixth LSTM layer with 16 neurons
    multivariate_lstm.add(layers.LSTM(units=16, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Seventh LSTM layer with 8 neurons
    multivariate_lstm.add(layers.LSTM(units=8, return_sequences=True))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))

    # Eighth LSTM layer with 4 neurons
    multivariate_lstm.add(layers.LSTM(units=4, return_sequences=False))
    multivariate_lstm.add(layers.Activation('relu'))
    multivariate_lstm.add(layers.Dropout(0.5))
    
    # Output layer with 1 neuron for the prediction of runoff
    multivariate_lstm.add(layers.Dense(units=1))
    
    # Compile the model using Adam optimizer and MSE as loss function
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    multivariate_lstm.compile(optimizer=optimizer, loss='mean_squared_error')
    
    return multivariate_lstm


In [28]:
def model_training(multivariate_lstm, X_train, y_train, X_test, y_test):
    # Define early stopping to avoid overfitting
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=50, restore_best_weights=True)

    # Train the model
    history = multivariate_lstm.fit(
        X_train, y_train, 
        epochs=1000, 
        batch_size=128, 
        validation_data=(X_test, y_test), 
        callbacks=[early_stopping],
        verbose=1  # Optional, to display training progress
    )
    
    # Return both the trained model and the training history
    return multivariate_lstm, history


In [29]:
def medel_predict(multivariate_lstm,  X_test, y_test):
	global predicted_values,X_test_predicted_df

	dataFrame = dataFrame_full[ :null_index_df["End"][i]]# Assuming the CSV file contains a 'Date' column
	# dataFrame['time'] = pd.to_datetime(dataFrame['time'])
	dataFrame.set_index('time', inplace=True)

	# Forecast Plot with Dates on X-axis
	predicted_values = multivariate_lstm.predict(X_test)

	d = {		
		'Actual_runoff': y_test.ravel(),
		'Predicted_runoff': predicted_values.ravel()
	}	
	d = pd.DataFrame(d)
	d.to_csv(f"F:\\geodata\\river_runoff_obs\\{name}_test{i}.csv")
	return d 

In [30]:
def forecast_model(multivariate_lstm,X_forecast):
	global forecast_df,forecast_values,forecast_arr
	forecast_values = multivariate_lstm.predict(X_forecast)
	# print(X_forecast[:,time_step-1,:X_train.shape[-1] - 1].shape,forecast_values[:,time_step-1].reshape(-1, 1).shape )
	forecast_arr = np.concatenate((X_forecast[:,time_step-1,:X_train.shape[-1]-1],forecast_values[:].reshape(-1, 1) ), axis=1)
	forecast_inversed = scaler.inverse_transform(forecast_arr)

	forecast_df = pd.DataFrame(forecast_inversed, columns=usecols[1:])
	# forecast_df = pd.DataFrame(pd.concat([X_forecast, forecast_values], axis=1), columns=usecols)
	dataFrame_full["dis"][null_index_df["Start"][i] : null_index_df["End"][i]+1]  = forecast_df['dis'][:len(forecast_values)] ###
	dataFrame_full.to_csv(f"F:\\geodata\\river_runoff_obs\\{name}_interpolated{i}.csv")
	forecast_df.to_csv(f"F:\\geodata\\river_runoff_obs\\{name}_forecast{i}.csv")
	forecast_df.plot()
	plt.savefig(f"F:\\geodata\\river_runoff_obs\\{name}_test{i}.svg")
	plt.show()
	return forecast_df
	


In [31]:
def model_evaluation(d):

	# Observed and predicted data
	observed = d['Actual_runoff']
	predicted = d['Predicted_runoff']
	
	# Correlation Coefficient (CC)
	CC = d['Predicted_runoff'].corr(d['Actual_runoff'])
	
	# Nash-Sutcliffe Efficiency (NSE)
	mean_observed = np.mean(observed)
	NSE = 1 - (np.sum((observed - predicted) ** 2) / np.sum((observed - mean_observed) ** 2))
	
	# Kling-Gupta Efficiency (KGE)
	mean_predicted = np.mean(predicted)
	std_observed = np.std(observed)
	std_predicted = np.std(predicted)
	r = np.corrcoef(observed, predicted)[0, 1]
	alpha = std_predicted / std_observed
	beta = mean_predicted / mean_observed
	KGE = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
	
	# Root-Mean-Square Error (RMSE)
	RMSE = np.sqrt(np.mean((predicted - observed) ** 2))
	
	# Mean Absolute Relative Error (MARE)
	MARE = np.mean(np.abs((observed - predicted) / mean_observed))
	
	# Percent Bias (PBIAS)
	PBIAS = 100 * np.sum(predicted - observed) / np.sum(observed)
	
	# Coefficient of Determination (R²)
	SS_res = np.sum((observed - predicted) ** 2)
	SS_tot = np.sum((observed - mean_observed) ** 2)
	R2 = 1 - (SS_res / SS_tot)
	
	# Print all metrics
	print("Correlation Coefficient (CC):", CC)
	print("Nash-Sutcliffe Efficiency (NSE):", NSE)
	print("Kling-Gupta Efficiency (KGE):", KGE)
	print("Root-Mean-Square Error (RMSE):", RMSE)
	print("Mean Absolute Relative Error (MARE):", MARE)
	print("Percent Bias (PBIAS):", PBIAS)
	print("Coefficient of Determination (R²):", R2)
	
	d.to_csv(f"F:\\geodata\\river_runoff_obs\\{name}_test{i}.csv")
	
	return CC,NSE,KGE,RMSE,MARE,PBIAS,R2

In [ ]:
for i in range(len(null_index_df)):
	print(i)
	intercept_data(i)
	df_scaled=scaler_data(dataFrame)
	X_train, y_train, X_test, y_test , X_forecast = split_data(df_scaled,i)
	multivariate_lstm = model_building(X_train)
	multivariate_lstm,history = model_training(multivariate_lstm, X_train, y_train, X_test, y_test)
	d=medel_predict(multivariate_lstm,  X_test, y_test)
	forecast_df  = forecast_model(multivariate_lstm,X_forecast)
	# d.missing_df(f"F:\geodata\\river_runoff_obs\\5_kq_imputMF{i}.csv")
	CC,NSE,KGE,RMSE,MARE,PBIAS,R2 = model_evaluation(d)
	
dataFrame_full.to_csv(f"F:\\geodata\\river_runoff_obs\\{name}_interpolated.csv")

0


C:\Users\HP\AppData\Local\Temp\ipykernel_70320\1391072043.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataFrame.drop(columns=['time'], inplace=True)
